# Bloom Monitoring Summer 2025

The `PlanetLabBloomMonitoring` class is a comprehensive pipeline designed for remote sensing-based monitoring of harmful algal blooms (HABs) using high-resolution Planet satellite imagery. It performs the following tasks:

## 1. Planet API Access
Authenticates with the Planet API using a valid API key to enable data querying and downloading.

## 2. Site Coordinate Definition
Reads and parses geographic coordinates of predefined water monitoring sites from the spreadsheet `Water Collection Sites 2025 V4.xlsx`.

## 3. Image Search and Filtering
Queries available PlanetScope (`PSScene`) satellite images based on:
- A user-provided acquisition date
- Acceptable cloud coverage threshold
- Standard-quality imagery
- Area of interest defined by geojson files

All matching image IDs are collected and filtered to ensure uniqueness.

## 4. Image Activation and Status Monitoring
Initiates activation requests for all selected image assets.  
Due to asynchronous activation, the class implements a delayed polling loop, checking each image's activation status every 2 minutes.  
This ensures all images transition from `activating` to `active` before proceeding to download.

## 5. Image Download and Mosaicking
Once activated, each image is downloaded as a `.tif` file.  
Images acquired on the same day are mosaicked into a single large `.tif` image and saved locally for visualization and processing.

## 6. Image Cropping by Site
Each mosaicked image is spatially cropped to a 300 × 300 pixel region centered around each water monitoring site.  
These cropped images are saved in site-specific subdirectories.

## 7. Bloom Detection via Pretrained CNN Model
A trained PyTorch-based CNN model (`planet_binary.pth`) is applied to the cropped images to predict bloom presence.  
The final output is a per-site, per-date prediction indicating whether a cyanobacterial bloom is likely present.


In [1]:
from utils import *

if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = '1c61ae19230448e19b5bc359cbf1232e'
    
session = requests.Session()
session.auth = (API_KEY, "")
from IPython import get_ipython

# sites_total = [
#     ('arrowhead_beach', 36.235731, -76.701847),
#     ('cannons_ferry', 36.272006, -76.675842),
#     ('rocky_hock_lane', 36.184133, -76.722858),
#     ('chowan_river', 36.055163, -76.69076),
#     ('point_comfort', 36.169908, -76.745905),
#     ('mt_gould', 36.125495, -76.740141),
#     ('mid_chowan_river', 36.20983, -76.72677),
#     ('north_chowan_river', 36.3236, -76.73354),
#     ('edenton_bay_dock', 36.055382, -76.610319),
#     ('edenhouse', 36.0476, -76.69611),
#     ('albemarle_sound', 35.99002, -76.6092)
# ]

/Users/sarah/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_excel('Water Collection Sites 2025 V4.xlsx', skiprows=[0,1])

ceeg_rows = df[df['Site'].str.startswith('CEEG', na=False)]

sites_total = [(row['Site'], row['lat'], row['long']) for _, row in ceeg_rows.iterrows()]
sites_total = sites_total[1:]
sites_total

[('CEEG 2', 36.169458, -76.751827),
 ('CEEG 3', 36.236102, -76.695387),
 ('CEEG 4', 36.055233, -76.686425),
 ('CEEG 5', 36.055924, -76.60952),
 ('CEEG 6', 36.183649, -76.71921),
 ('CEEG 7', 36.1432, -76.7539),
 ('CEEG 8', 36.123866, -76.74263),
 ('CEEG 9', 36.0430926748227, -76.7084442758706)]

In [14]:
class PlanetLabBloomMonitoring:
    """
    PlanetLabBloomMonitoring is designed for downloading and monitoring harmful algal blooms (HABs)
    using high-resolution Planet satellite imagery during Summer 2025. Updates to come.
    
    Attributes:
    -----------
    json_dir : str
        Directory to geojson file of entire Chowan River
    download_path : str
        Directory path where daily images will be downloaded
    model_path : str
        Path to trained bloom classification model
    cc : float
        Maximum allowable cloud cover percentage
    input_date: str
        Date of interest
    api_key: str
        Planet API key for authentication
    item_type: str
        The item type to query (e.g., "PSScene")
    """
    def __init__(self, json_dir, download_path, model_path, cc, input_date, api_key):
        self.json_dir = json_dir
        self.download_path = download_path
        self.model_path = model_path
        self.cc = cc
        self.input_date = input_date
        self.api_key = api_key
        self.item_type = "PSScene"
        self.image_ids_set = set()

    def get_date_filters(self):
        date_obj = datetime.strptime(self.input_date, "%Y-%m-%d")
        date_2_days_prior = date_obj - timedelta(days=1)
        return {
            "gte": date_2_days_prior.strftime('%Y-%m-%dT') + '00:00:00.000Z',
            "lte": date_obj.strftime('%Y-%m-%dT') + '00:00:00.000Z'
        }

    def build_search_request(self, geometry, date_filters):
        return {
            "item_types": [self.item_type],
            "filter": {
                "type": "AndFilter",
                "config": [
                    {"type": "GeometryFilter", "field_name": "geometry", "config": geometry},
                    {"type": "DateRangeFilter", "field_name": "acquired", "config": date_filters},
                    {"type": "RangeFilter", "field_name": "cloud_cover", "config": {"lte": self.cc}},
                    {"type": "StringInFilter", "field_name": "quality_category", "config": ["standard"]}
                ]
            }
        }

    def fetch_image_ids(self):
        date_filters = self.get_date_filters()
        for file in os.listdir(self.json_dir):
            if file.endswith('.geojson'):
                with open(os.path.join(self.json_dir, file), 'r') as f:
                    data = json.load(f)
                    geometry = {
                        "type": "Polygon",
                        "coordinates": data['features'][0]['geometry']['coordinates']
                    }
                    search_request = self.build_search_request(geometry, date_filters)
                    response = requests.post(
                        'https://api.planet.com/data/v1/quick-search',
                        auth=HTTPBasicAuth(self.api_key, ''),
                        json=search_request
                    )
                    results = response.json()
                    ids = [feature['id'] for feature in results.get('features', [])]
                    self.image_ids_set.update(ids)
        print("Total unique image_ids:", len(self.image_ids_set))

    def activate_and_get_download_links(self):
        download_links = {}
        for image_id in self.image_ids_set:
            id_url = f'https://api.planet.com/data/v1/item-types/{self.item_type}/items/{image_id}/assets'
            result = requests.get(id_url, auth=HTTPBasicAuth(self.api_key, ''))
            asset_info = result.json().get("ortho_visual")

            if asset_info:
                links = asset_info["_links"]
                requests.get(links["activate"], auth=HTTPBasicAuth(self.api_key, ''))
                status = requests.get(links["_self"], auth=HTTPBasicAuth(self.api_key, '')).json()
                download_link = status.get("location")
                if download_link:
                    download_links[image_id] = download_link
                else:
                    print(f"No download location for image ID {image_id}")
        return download_links

    def run(self, sites_total):
        self.fetch_image_ids()
        self.imagelinks_delayed(self.image_ids_set, self.item_type, self.api_key, max_retries=10)  # Optional external step
        download_links = self.activate_and_get_download_links()
        self.image_download(self.download_path, download_links)
        self.plot_large_image(self.download_path)
        self.crop_and_download(500, 500, self.download_path, sites_total)
        self.bloom_model_implementation(self.download_path, self.model_path)


    def imagelinks_delayed(self, image_ids_set, item_type, api_key, max_retries=10):
        activation_links = {}
        self_links = {}

        # Step 1: Request activation for all images
        for image_id in list(image_ids_set):
            try:
                id_url = f'https://api.planet.com/data/v1/item-types/{item_type}/items/{image_id}/assets'
                response = requests.get(id_url, auth=HTTPBasicAuth(api_key, ''))
                result = response.json()

                links = result["ortho_visual"]["_links"]
                activation_links[image_id] = links["activate"]
                self_links[image_id] = links["_self"]

                # Trigger activation
                requests.get(activation_links[image_id], auth=HTTPBasicAuth(api_key, ''))
                print(f"Activation requested for {image_id}")
            except Exception as e:
                print(f"Error requesting activation for {image_id}: {e}")
                continue

        # Step 2: Poll status for each image
        for attempt in range(max_retries):
            print(f"\n--- Polling attempt {attempt + 1} ---")
            remaining_ids = list(self_links.keys())
            for image_id in remaining_ids:
                try:
                    status_resp = requests.get(self_links[image_id], auth=HTTPBasicAuth(api_key, ''))
                    status = status_resp.json().get("status", "")
                    print(f"Status of {image_id}: {status}")

                    if status != "activating":
                        # Remove from further polling
                        self_links.pop(image_id)
                except Exception as e:
                    print(f"Error checking activation status for {image_id}: {e}")

            if not self_links:
                print("All images activated.")
                break

            print(f"Waiting 2 minutes before next polling round...")
            time.sleep(120)

        if self_links:
            print(f"The following images did not activate in time: {list(self_links.keys())}")



    def image_download(self, download_path, download_links):
        if not (os.path.isdir(self.download_path)):
            print('Figure directory didn''t exist, creating now.')
            os.mkdir(self.download_path)
        else:
            print('Figure directory exists.') 

        for image_name, download_link in download_links.items():
            try:
                # Use allow_redirects=False to prevent automatic redirects
                response = requests.get(download_link, stream=True, allow_redirects=True)

                if response.status_code == 302:  # 302 indicates a redirect

                    final_url = response.headers.get('Location')

                    if final_url:
                        response = requests.get(final_url, stream=True)
                    else:
                        print(f"Failed to retrieve {image_name}. Redirect URL not found.")
                        continue

                if response.status_code == 200:
                    local_file_path = f'./{self.download_path}/{image_name}.tif'

                    with open(local_file_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)

                    print(f"Downloaded {image_name} as {local_file_path}")
                else:
                    print(f"Failed to retrieve {image_name}. Status code: {response.status_code}")
            except Exception as e:
                print(f"An error occurred while downloading {image_name}: {e}")
            

    def plot_large_image(self, download_path):
        dategroups = defaultdict(list)
        tif_files = glob.glob(os.path.join(self.download_path, '*.tif'))
        for tif_file in tif_files:
            filename = os.path.basename(tif_file)
            date = filename.split('_')[0]
            dategroups[date].append(tif_file)

        for date, files in dategroups.items():
            src_files_to_mosaic = []
            for tif_file in files:
                src = rasterio.open(tif_file)
                src_files_to_mosaic.append(src)
                print(src_files_to_mosaic)

            mosaic, out_transform = merge(src_files_to_mosaic)

            out_meta = src_files_to_mosaic[0].meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": out_transform
            })
            output_path = f'mosaic_{date}.tif'
            with rasterio.open(output_path, "w", **out_meta) as dest:
                dest.write(mosaic)

            for src in src_files_to_mosaic:
                src.close()

    def crop_and_download(self, x, y, download_path, sites_total):
        for fi in tqdm(sorted(os.listdir(self.download_path))):
            if fi.endswith(".tif"):
                working_path = os.path.join(self.download_path, fi)
                with rasterio.open(working_path) as rds:
                    transformer = Transformer.from_crs("EPSG:4326", rds.crs, always_xy=True)

                    for site_name, lat, lon in sites_total:
                        xx, yy = transformer.transform(lon, lat)
                        row, col = rds.index(xx, yy)

                        if row > x and col > y:
                            try:
                                window = Window.from_slices(rows=(row - x, row + x), cols=(col - y, col + y))
                                data = rds.read(window=window)
                                data = np.moveaxis(data, 0, 2) 

                                if data.shape[0] != 2 * x or data.shape[1] != 2 * y: continue

                                black_space = np.mean(data / 255)

                                if black_space < 0.25 or black_space >= 0.9: continue

        #                         plt.imshow(data)
        #                         plt.axis('off')
        #                         plt.show()
                                out_dir = os.path.join(self.download_path, site_name)
                                os.makedirs(out_dir, exist_ok=True)

                                out_path = os.path.join(out_dir, fi)
                                im = Image.fromarray(data.astype(np.uint8))
                                im.save(out_path)

                            except Exception as e:
                                print(f"{site_name} in {fi} failed: {e}")
                                continue

    def bloom_model_implementation(self, download_path, model_path):
        dataset = DailyMonitoringDataset(root_dir=self.download_path)
        date = DataLoader(dataset, batch_size=1, shuffle = False)
        model = CNN()
        model.load_state_dict(torch.load(self.model_path, 
        map_location=torch.device('cpu')))

        model.eval()
        tracking = {'date': [],'loc':[], 'pred': []}
        with torch.no_grad():
            for i, batch in tqdm(enumerate(date)):
                x = batch['image'].float()
                p = batch['file_path'] 

                output = model(x)
        #         logits = (output.cpu().detach().numpy()).astype(int)
                output_binary = (output >= 0)

                dates = [datetime.strptime(fp.split('/')[-1][:8], "%Y%m%d").strftime("%m-%d-%Y") for fp in p]
                locs = [fp.split('/')[-2] for fp in p]
                tracking['date'].extend(dates) 
                tracking['loc'].extend(locs)
                tracking['pred'].extend(output_binary.reshape(-1).tolist())  
        print(tracking)

# USER INTERFACE BELOW
## ENTER CLOUD COVERAGE IN DECIMAL
## ENTER DATE OF INTEREST IN FORMAT YYYY-MM-DD

In [11]:
cc = float(input("Cloud coverage: ").strip())
input_date = input("Date of interest in format YYYY-MM-DD").strip()
# download_path='may27'
model_path="planet_binary.pt"
downloader = PlanetLabBloomMonitoring(
    json_dir="jsons",
    download_path=input_date,
    model_path=model_path,
    cc=cc,
    input_date=input_date,
    api_key=API_KEY
)

downloader.run(sites_total)


Cloud coverage: 0.9
Date of interest in format YYYY-MM-DD2025-05-21
